**Swapping providers:** `LiteLLMClient` is provider-agnostic — change only the model string.
- local Ollama: `LiteLLMClient("ollama/qwen2.5:7b", api_base="http://localhost:11434")`
- OpenAI: `LiteLLMClient("gpt-4o-mini", api_key=...)`
- Anthropic: `LiteLLMClient("claude-3-5-haiku-latest", api_key=...)`
- Gemini: `LiteLLMClient("gemini/gemini-1.5-flash", api_key=...)`

Install the LLM backend with `pip install -e ".[llm]"`.

In [ ]:
import pandas as pd, numpy as np, torch
from sentence_transformers import SentenceTransformer
from book_recsys.llm.retrieve import Retriever
from book_recsys.llm.clients import LiteLLMClient
from book_recsys.models.llm.recommender import LLMRecommender
from book_recsys.features.document import build_documents

In [ ]:
catalog = pd.read_parquet("artifacts/catalog.parquet")
emb = np.load("artifacts/embeddings.npy")
book_ids = catalog["book_id"].tolist()
id_to_doc = dict(zip(book_ids, build_documents(catalog)))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
encoder = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
retriever = Retriever(book_ids, emb, encoder=encoder)
rec = LLMRecommender(retriever, id_to_doc,
                      LiteLLMClient("ollama/qwen2.5:7b", api_base="http://localhost:11434")).fit()

In [ ]:
rec.recommend("a comforting cozy mystery for my grandmother", k=10)

In [ ]:
rec.recommend(book_ids[:3], k=10)

In [ ]:
rec.recommend({"history": book_ids[:3], "query": "something darker than usual"}, k=10)

**Swapping providers:** `LiteLLMClient` is provider-agnostic — change only the model string.
- local Ollama: `LiteLLMClient("ollama/qwen2.5:7b", api_base="http://localhost:11434")`
- OpenAI: `LiteLLMClient("gpt-4o-mini", api_key=...)`
- Anthropic: `LiteLLMClient("claude-3-5-haiku-latest", api_key=...)`
- Gemini: `LiteLLMClient("gemini/gemini-1.5-flash", api_key=...)`

Install the LLM backend with `pip install -e ".[llm]"`.